# **Tiamat - Previsão de Renda Anual Superior ou Igual a 50 Mil Dólares utilizando Machine Learning** 

Autora: Gabriela O. de Lima Cabral

Professor: Daniel Roberto Cassar

Ilum - Escola de ciência

Centro Nacional de Pesquisa em Energia e Materiais (CNPEM)

## **1. Introdução**

Decisões relacionadas à concessão de crédito, contratação de profissionais, distribuição de benefícios e formulação de políticas públicas dependem, em grande medida, de estimativas sobre a renda da população. Nesse contexto, os censos desempenham um papel fundamental, pois permitem compreender como a renda está distribuída e registram informações demográficas, educacionais e ocupacionais dos indivíduos. A partir desses dados, surge a seguinte questão: é possível prever a faixa de renda de uma pessoa com base em suas características pessoais e profissionais?

Neste trabalho, abordaremos esse problema por meio de técnicas de aprendizado de máquina. Dado um conjunto de atributos de um indivíduo, como idade, escolaridade, ocupação, estado civil, horas trabalhadas por semana e ganhos de capital, será desenvolvido um modelo capaz de prever se sua renda anual é superior a 50 mil dólares. Esse limiar é amplamente utilizado em estudos socioeconômicos e caracteriza uma classe minoritária no conjunto de dados. Como consequência, a distribuição das classes é desbalanceada, tornando a acurácia tradicional uma métrica potencialmente enganosa e justificando o uso da acurácia balanceada como principal medida de avaliação.

O objetivo geral deste trabalho é comparar diferentes algoritmos de aprendizado supervisionado nessa tarefa, realizando ajuste sistemático de hiperparâmetros e avaliando sua capacidade de generalização para dados não vistos. Para isso, serão comparados seis algoritmos pertencentes a famílias distintas: regressão logística, k-vizinhos mais próximos (KNN), árvore de decisão, floresta aleatória, gradient boosting e máquina de vetores de suporte (SVM). Como referência, será utilizado um modelo baseline que sempre prevê a classe majoritária. Os hiperparâmetros de cada algoritmo, bem como etapas de pré-processamento, incluindo normalização de atributos numéricos, codificação de variáveis categóricas e redução de dimensionalidade, serão otimizados por meio da avaliação de 100 configurações distintas para cada método. Posteriormente, será investigado se as diferenças de desempenho observadas são estatisticamente significativas por meio de validação cruzada repetida. Por fim, o modelo selecionado será avaliado uma única vez em um conjunto de teste reservado desde o início do estudo, verificando sua capacidade de capturar padrões reais dos dados e analisando a uniformidade de desempenho entre diferentes grupos de sexo e raça.

## 2. O conjunto de dados

O conjunto de dados utilizado é o Adult, também conhecido como Census Income, disponível no UCI Machine Learning Repository e derivado de uma extração da Current Population Survey de 1994 (Becker e Kohavi, 1996). O arquivo de treino original tem 32.561 registros, dos quais 30.139 permanecem após a remoção de valores ausentes e de duplicatas exatas. Cada registro traz 13 atributos utilizados pelo modelo, cinco numéricos e oito categóricos. O peso amostral do censo, `fnlwgt`, foi deliberadamente excluído, porque descreve a representatividade do registro na população e não uma característica da pessoa. Esse conjunto é amplamente estudado, o que permite comparar os resultados com a literatura, e reúne atributos mistos, valores ausentes e forte assimetria, o que aproxima o trabalho de situações reais. Ao mesmo tempo, é um conjunto antigo: os registros refletem o mercado de trabalho e a estrutura social dos Estados Unidos de 1994, e não devem ser lidos como retrato do presente.

O arquivo usado neste projeto é o **`adult.data`**, com **32.561 registros**.Cada registro descreve uma pessoa por 14 atributos e pela renda anual, que é o alvo. O UCI traz também o arquivo `adult.test` com 16.281 registros, ele não é usado aqui, porque a divisão treino/teste do projeto é feita a partir do `adult.data`, de forma estratificada e controlada pela semente.

### 2.1 Dicionário dos atributos

| Atributo | Tipo | Descrição |
|---|---|---|
| `age` | numérico | Idade em anos |
| `workclass` | categórico | Tipo de vínculo de trabalho |
| `fnlwgt` | numérico | Peso amostral do censo (não é usado como atributo, ver seção 4.1) |
| `education` | categórico | Nível de escolaridade |
| `educational-num` | numérico | Anos de estudo |
| `marital-status` | categórico | Estado civil |
| `occupation` | categórico | Ocupação |
| `relationship` | categórico | Papel na família |
| `race` | categórico | Raça autodeclarada |
| `gender` | categórico | Sexo |
| `capital-gain` | numérico | Ganho de capital no ano |
| `capital-loss` | numérico | Perda de capital no ano |
| `hours-per-week` | numérico | Horas trabalhadas por semana |
| `native-country` | categórico | País de origem |
| `income` | alvo | `>50K` ou `<=50K` |

Os valores ausentes aparecem como `?` e ocorrem em `workclass`, `occupation` e `native-country`. A classe positiva, renda acima de 50 mil dólares, é minoritária, com cerca de um quarto dos registros.

# Bibliotecas e requerimentos

Todas as etapas com aleatoriedade usam a semente 24. As versões das bibliotecas estão fixadas em `requirements.txt`, para que o resultado possa ser reproduzido.

In [1]:
import sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
import shap
import sklearn
from scipy import stats
from optuna.trial import TrialState
from sklearn.dummy import DummyClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.metrics import (ConfusionMatrixDisplay, RocCurveDisplay, balanced_accuracy_score,
                             confusion_matrix, make_scorer, matthews_corrcoef, roc_auc_score)
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, cross_validate